In [ ]:
!pip install -q streamlit langchain langchain-google-genai python-dotenv requests pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [ ]:
# Create the .env file to store the API key
%%writefile .env
GOOGLE_API_KEY=YOUR_GOOGLE_API_KEY
#ENTER YOUR GOOGLE API KEY

Writing .env


In [ ]:
!pip install --upgrade -q langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 21.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colabsqlviz 0.2.1 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.5 which is incompatible.


In [ ]:
%%writefile agent.py
import os
import datetime
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import initialize_agent, Tool, AgentType

# --- MODIFIED: Load environment variables once ---
load_dotenv()

# --- MODIFIED: Initialize the LLM once ---
llm = ChatGoogleGenerativeAI(
    model='models/gemini-1.5-pro-002',
    temperature=0.5,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

# --- MODIFIED & UPGRADED: Dummy Database for Orders ---
# This database is now more detailed to support new features like proactive delay
# notifications and multi-item returns.
orders_db = {
    "12345": {
        "status": "delayed",
        "eta": "August 12, 2025",
        "items": [
            {"name": "Smart Watch", "quantity": 1},
            {"name": "Wireless Earbuds", "quantity": 1}
        ],
        "shipping_address": "Hyderabad, Telangana"
    },
    "98765": {
        "status": "delivered",
        "eta": "August 1, 2025",
        "items": [{"name": "Blue T-Shirt", "quantity": 1}],
        "shipping_address": "Mumbai, Maharashtra"
    },
    "55555": {
        "status": "shipped",
        "eta": "August 9, 2025",
        "items": [{"name": "Pro-Gamer Mouse", "quantity": 1}],
        "shipping_address": "Bengaluru, Karnataka"
    }
}

# +++ NEW FEATURE: Dummy Database for Product Catalog +++
# This new database simulates a product catalog to answer product-related questions.
product_catalog = {
    "smart watch": {
        "available_colors": ["Black", "Silver", "Rose Gold"],
        "description": "A next-gen smartwatch with heart rate monitoring, GPS, and a 5-day battery life.",
        "price": "₹12,500"
    },
    "wireless earbuds": {
        "available_colors": ["White", "Black"],
        "description": "High-fidelity wireless earbuds with active noise cancellation and 18-hour playtime with case.",
        "price": "₹4,999"
    },
    "blue t-shirt": {
        "available_colors": ["Blue", "Red", "Green"],
        "description": "A 100% cotton t-shirt, available in multiple sizes.",
        "price": "₹999"
    },
     "pro-gamer mouse": {
        "available_colors": ["Black with RGB"],
        "description": "An ergonomic gaming mouse with 16,000 DPI, 8 programmable buttons, and customizable RGB lighting.",
        "price": "₹3,500"
    }
}


# --- MODIFIED & UPGRADED: get_order_status function ---
# Now provides more detailed information and returns a dictionary for richer display in the UI.
def get_order_status(order_id: str):
    """Check order status by order ID"""
    order_id = order_id.strip()
    order = orders_db.get(order_id)
    if not order:
        return f"❌ No order found for ID {order_id}"

    # Return a dictionary instead of just a string
    return {
        "tool": "OrderStatusTool",
        "order_id": order_id,
        "status": order['status'],
        "eta": order['eta'],
        "items": [f"{item['quantity']}x {item['name']}" for item in order['items']]
    }


# --- MODIFIED & UPGRADED: return_item function ---
# This tool is now smarter and can handle multi-item orders.
def return_item(query: str):
    """
    Initiate a return for an order ID. If an order has multiple items,
    it can also handle returning a specific item from that order.
    Example: 'return item Smart Watch from order 12345'
    """
    query = query.lower().strip()
    # Basic parsing to find order ID
    order_id = ''.join(filter(str.isdigit, query))

    if not order_id or order_id not in orders_db:
        return f"❌ To initiate a return, please provide a valid order ID."

    order = orders_db.get(order_id)
    items_in_order = [item['name'].lower() for item in order['items']]

    # Check if a specific item was mentioned for return
    item_to_return = None
    for item_name in items_in_order:
        if item_name in query:
            item_to_return = item_name
            break

    # If no specific item is mentioned and there are multiple items, ask for clarification.
    if not item_to_return and len(items_in_order) > 1:
        item_list = ", ".join([item['name'] for item in order['items']])
        return f"🤔 Order #{order_id} has multiple items: {item_list}. Which item would you like to return?"

    # If only one item or a specific item was mentioned
    return_item_name = item_to_return or items_in_order[0]
    return (
        f"✅ Return initiated for '{return_item_name.title()}' from Order ID {order_id}. "
        f"Your return label is ready: https://dummylabel.com/{order_id}"
    )

# +++ NEW FEATURE: get_product_details function +++
# This new tool retrieves information from the product catalog.
def get_product_details(product_name: str):
    """Get details about a specific product from the catalog."""
    product = product_catalog.get(product_name.lower().strip())
    if not product:
        return f"🔎 I couldn't find any details for a product named '{product_name}'."
    details = (
        f"**{product_name.title()}**\n"
        f"- **Description**: {product['description']}\n"
        f"- **Price**: {product['price']}\n"
        f"- **Available Colors**: {', '.join(product['available_colors'])}"
    )
    return details

# --- Original Function (No Changes) ---
def get_return_policy(_: str = ""):
    """Get return policy information"""
    return (
        "📜 **Return Policy:**\n"
        "- Return within 7 days of delivery.\n"
        "- Product must be unused and in its original packaging.\n"
        "- Refund will be processed within 5 business days after we receive the item."
    )

# --- MODIFIED: Tool list now includes the new ProductInquiryTool ---
tools = [
    Tool(
        name="OrderStatusTool",
        func=get_order_status,
        description="Use this tool to check the status of an order using an order ID."
    ),
    Tool(
        name="ReturnTool",
        func=return_item,
        description="Use this tool to initiate a return for an order ID. You can also specify which item to return from an order."
    ),
    Tool(
        name="ReturnPolicyTool",
        func=lambda q: get_return_policy(q),
        description="Use this tool to get the company's return policy."
    ),
    # +++ NEW FEATURE: Adding the new tool to the agent's skillset +++
    Tool(
        name="ProductInquiryTool",
        func=get_product_details,
        description="Use this tool to get details, price, and colors for a specific product."
    )
]

# --- MODIFIED: Agent initialization with a more descriptive system message ---
# This helps the agent better understand its role and how to interact with the user.
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    agent_kwargs={
        "prefix": """
        You are E-Comm Assist, a friendly and helpful AI assistant for an e-commerce store.
        Your goal is to assist customers with their orders, returns, and product questions.
        Be polite and provide clear, concise answers.
        Today's date is {date}.
        """.format(date=datetime.date.today().strftime("%B %d, %Y"))
    }
)

# --- Main function to run the agent (No Changes to logic) ---
def run_agent(query: str):
    return agent.run(query)




Writing agent.py


In [ ]:
%%writefile app.py
import streamlit as st
from agent import run_agent, orders_db

# --- MODIFIED: Page configuration ---
st.set_page_config(page_title="E-Comm Assist Pro", layout="centered")
st.title("📦 E-Comm Assist Pro")

# --- MODIFIED: Instructions updated with the new product inquiry feature ---
st.markdown("""
Welcome! I'm your AI assistant. You can ask me about:
- **📬 Order Tracking**: e.g., "Where is order 12345?"
- **🔁 Returns**: e.g., "I want to return order 98765" or "return the smart watch from order 12345"
- **📜 Return Policies**: e.g., "What is your return policy?"
- **🛍️ Product Info**: e.g., "Tell me about the smart watch"
""")

# +++ NEW FEATURE: Proactive Delay Notification +++
# The agent now checks for delayed orders and informs the user upfront.
st.header("Alerts")
has_delayed_orders = False
for order_id, details in orders_db.items():
    if details['status'] == 'delayed':
        st.warning(f"🔔 **Heads up!** Order #{order_id} is delayed. The new ETA is {details['eta']}.")
        has_delayed_orders = True
if not has_delayed_orders:
    st.info("✅ All your orders are currently on schedule.")

# --- User Input Section (No Changes) ---
st.header("How can I help you today?")
user_query = st.text_input("💬 Ask me anything about your orders or our products...")

if st.button("Ask"):
    if user_query.strip():
        with st.spinner("🤖 Thinking..."):
            try:
                response = run_agent(user_query)

                # +++ NEW FEATURE: Richer, Interactive Responses +++
                # The app now checks the response type to provide better UI.
                st.success("✅ Here's your answer:")
                if isinstance(response, dict) and response.get("tool") == "OrderStatusTool":
                    st.markdown(f"### Order Status for #{response['order_id']}")
                    st.markdown(f"**Status**: {response['status'].title()}")
                    st.markdown(f"**ETA**: {response['eta']}")
                    st.markdown(f"**Items**: {', '.join(response['items'])}")

                    # Add a visual progress bar based on status
                    if response['status'] == 'shipped':
                        st.progress(50, text="On its way!")
                    elif response['status'] == 'delivered':
                        st.progress(100, text="Delivered!")
                    elif response['status'] == 'delayed':
                        st.progress(25, text="Delayed")

                else:
                    # For all other text-based responses
                    st.markdown(response)

            except Exception as e:
                st.error(f"❌ An error occurred: {str(e)}")
    else:
        st.warning("Please enter your question.")


Writing app.py


In [ ]:
!ngrok config add-authtoken (GIVE YOUR NGROK AUTHTOKEN)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
# 🌐 Launch Streamlit with pyngrok (Corrected with Authentication)
from pyngrok import ngrok
from getpass import getpass # <--- Import getpass
import threading
import time
import os

# --- Authenticate ngrok using a secure prompt ---
try:
    authtoken = "303bJRuA4W5iprtBXqmqmUIFXhX_7ZKVyCvGnxEEGrdrYNMx6" # <--- Securely prompt for the token
    ngrok.set_auth_token(authtoken)
    print("✅ ngrok authenticated successfully for this session!")
except Exception as e:
    print(f"❌ An error occurred during authentication.")
    # Stop execution if authentication fails
    raise e

# --- Run Streamlit in a separate thread ---
def run_streamlit():
    os.system('streamlit run app.py')

# Start the Streamlit app in the background
thread = threading.Thread(target=run_streamlit)
thread.start()

# Give Streamlit a few seconds to start up
time.sleep(5)

# --- Connect to ngrok and get the public URL ---
public_url = ngrok.connect(addr="8501", proto="http")
print(f'🔗 Public App URL: {public_url}')

✅ ngrok authenticated successfully for this session!
🔗 Public App URL: NgrokTunnel: "https://e4ca0b1dfa36.ngrok-free.app" -> "http://localhost:8501"
